# Agent training
Train a reinforcement-learning agent **directly on the FMU** of your building with `fmi_gym`: fast, no MQTT, no gateway.

* The FMU and `parameters.py` (generated by the FMU export) live in `/home/jovyan/field` (read-only). This notebook **reads** them, it never writes there.
* The training runs in a **temporary folder**: the FMU is copied there and EnergyPlus writes its `Output_EPExport_Slave/` there, so nothing is left in your Workspace except the model you save.
* The model is saved in `ai/models/` (next to `ai/controller.py`) and loaded from your controller, see the last section.

The default training budget is small (a couple of episodes) so the notebook runs in a minute or two; raise `TOTAL_TIMESTEPS` once everything works.

In [ ]:
import importlib.util, json, os, shutil, tempfile
import numpy as np
import gym
import gymnasium
from stable_baselines3 import PPO
from fmi_mlc import fmi_gym

FIELD_DIR = "/home/jovyan/field"
MODEL_DIR = "/home/jovyan/notebooks/ai/models"     # = ai/models/ in your Workspace
MODEL_NAME = "my_agent"
TOTAL_TIMESTEPS = int(os.getenv("TOTAL_TIMESTEPS", 2000))   # raise it for a real training (one episode = the whole simulated period)
os.makedirs(MODEL_DIR, exist_ok=True)

# parameters.py is loaded by path: it is in the read-only field folder, not next to this notebook
spec = importlib.util.spec_from_file_location("field_parameters", os.path.join(FIELD_DIR, "parameters.py"))
field_parameters = importlib.util.module_from_spec(spec)
spec.loader.exec_module(field_parameters)
PARAMS = dict(field_parameters.parameter)
FMU_SOURCE = os.path.join(FIELD_DIR, PARAMS["fmu_path"])     # the controlled FMU
print("FMU:", FMU_SOURCE, "| exists:", os.path.exists(FMU_SOURCE))

## About warm-up
`fmu_warmup_time` is an **absolute** simulation time (seconds from the start of the year), not a duration.
During the warm-up `fmi_gym.reset()` steps the FMU from `fmu_start_time` to `fmu_warmup_time` forcing the action to `[0]*n`,
so your agent does not control anything in that period; the episode starts at `fmu_warmup_time`.
(EnergyPlus has its own, separate warm-up days inside the FMU.)

In [ ]:
print({k: PARAMS.get(k) for k in ("fmu_step_size", "fmu_start_time", "fmu_warmup_time", "fmu_final_time",
                                   "action_names", "action_min", "action_max", "observation_names", "reward_names")})
steps_per_episode = (PARAMS["fmu_final_time"] - (PARAMS.get("fmu_warmup_time") or PARAMS["fmu_start_time"])) // PARAMS["fmu_step_size"]
print("steps per episode:", steps_per_episode, "-> episodes in the budget:", round(TOTAL_TIMESTEPS / steps_per_episode, 2))

## What the agent sees, and what it is rewarded for
The observation is the list `observation_names` (FMU outputs). The order matters: **`ai/controller.py` must build the observation vector in exactly this order**.

If your FMU export has no observation role yet, `observation_names` is empty and a Gym environment cannot be built.
TODO: choose the observations below (FMU variable names, see `points.yaml`). The default observes only the zone temperature, which is legitimate but poor
(no time of day, no outdoor temperature: add what a controller could really measure).

`fmi_gym` adds up the variables in `reward_names`, which is rarely what you want (it would add Joules and degrees). Define your own reward from the FMU values; the variables the reward needs but the agent does not see are declared as *hidden observations*.

In [ ]:
ENERGY_VAR = "Electricity_Facility"                 # J per step
TEMP_VAR = "Zone_Mean_Air_Temperature_KITCHEN"      # degC
COMFORT_MIN = 20.0

# TODO: choose the observations (FMU variable names) -- the order is the order of the observation vector
if not PARAMS["observation_names"]:
    PARAMS["observation_names"] = [TEMP_VAR]
# variables the reward needs but the agent does not see: "hidden" observations (the FMU is asked for them, the agent is not given them)
PARAMS["hidden_observation_names"] = [v for v in (ENERGY_VAR, TEMP_VAR) if v not in PARAMS["observation_names"]]
PARAMS["reward_names"] = []        # the reward is computed by reward_fn below, not summed by fmi_gym
n_obs = len(PARAMS["observation_names"])
PARAMS["observation_min"] = np.full(n_obs, -1e12)
PARAMS["observation_max"] = np.full(n_obs, 1e12)
print("observation order:", PARAMS["observation_names"], "| hidden:", PARAMS["hidden_observation_names"])

# TODO: reward. It receives the dict {FMU variable: value} of the step.
def reward_fn(v):
    energy_kwh = v[ENERGY_VAR] / 3.6e6
    discomfort = max(0.0, COMFORT_MIN - v[TEMP_VAR])
    return -energy_kwh - 1.0 * discomfort           # TODO: tune the weights

## Environment
`fmi_gym` follows the old `gym` API (`reset() -> obs`, `step() -> obs, reward, done, info`) while stable_baselines3 2.x expects `gymnasium`.
The adapter below bridges the two, and replaces the reward with `reward_fn` (`reward_fn=None` keeps the `fmi_gym` reward).

In [ ]:
class GymnasiumAdapter(gymnasium.Env):
    def __init__(self, env, reward_fn=None):
        self.env = env
        self.reward_fn = reward_fn
        self.action_space = gymnasium.spaces.Box(env.action_space.low, env.action_space.high, dtype=np.float32)
        self.observation_space = gymnasium.spaces.Box(env.observation_space.low, env.observation_space.high, dtype=np.float32)
    def reset(self, *, seed=None, options=None):
        return np.asarray(self.env.reset(), dtype=np.float32), {}
    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        if self.reward_fn is not None:
            row = {k: list(v.values())[0] for k, v in json.loads(info["data"]).items()}
            reward = self.reward_fn(row)
        return np.asarray(obs, dtype=np.float32), float(reward), bool(done), False, info
    def close(self):
        self.env.close()

## Training
Everything that touches the FMU runs in a fresh temporary folder: the `try/finally` brings you back to your folder and removes it, even if the training fails.
The FMU path in `PARAMS` is the absolute path of the copy.

In [ ]:
# TODO: choose the algorithm and its hyper-parameters (n_steps = steps collected between two policy updates)
def make_model(env):
    return PPO("MlpPolicy", env, n_steps=256, batch_size=64, verbose=1, seed=PARAMS["seed"])

workdir = tempfile.mkdtemp(prefix="agent_training_")
start_dir = os.getcwd()
env = None
try:
    shutil.copy(FMU_SOURCE, workdir)
    os.chdir(workdir)                                   # EnergyPlus writes Output_EPExport_Slave/ in the cwd
    PARAMS["fmu_path"] = os.path.join(workdir, os.path.basename(FMU_SOURCE))
    env = GymnasiumAdapter(fmi_gym(PARAMS), reward_fn=reward_fn)

    model = make_model(env)
    model.learn(total_timesteps=TOTAL_TIMESTEPS)
    model.save(os.path.join(MODEL_DIR, MODEL_NAME))     # -> ai/models/my_agent.zip
    print("saved", os.path.join(MODEL_DIR, MODEL_NAME) + ".zip")
finally:
    if env is not None:
        env.close()
    os.chdir(start_dir)
    shutil.rmtree(workdir, ignore_errors=True)

## Evaluation
Run the trained policy (deterministic) for one episode on a fresh copy of the FMU and compare it with a fixed setpoint.
TODO: extend it (more baselines, comfort-violation hours, kWh over the period, plots of `info["data"]`).

In [ ]:
def run_episode(policy):
    # policy(obs) -> action. Same temporary-folder pattern as for the training.
    wd = tempfile.mkdtemp(prefix="agent_eval_")
    cwd = os.getcwd()
    e = None
    try:
        shutil.copy(FMU_SOURCE, wd)
        os.chdir(wd)
        p = dict(PARAMS, fmu_path=os.path.join(wd, os.path.basename(FMU_SOURCE)))
        e = GymnasiumAdapter(fmi_gym(p), reward_fn=reward_fn)
        obs, _ = e.reset()
        total, done, n = 0.0, False, 0
        while not done:
            obs, r, done, _, _ = e.step(policy(obs))
            total += r
            n += 1
        return total, n
    finally:
        if e is not None:
            e.close()
        os.chdir(cwd)
        shutil.rmtree(wd, ignore_errors=True)

low, high = PARAMS["action_min"], PARAMS["action_max"]
fixed = lambda obs: np.full(len(low), 21.0).clip(low, high).astype(np.float32)
agent = lambda obs: model.predict(obs, deterministic=True)[0]

print("fixed setpoint 21 degC : return = %.2f over %d steps" % run_episode(fixed))
print("trained agent          : return = %.2f over %d steps" % run_episode(agent))

## Using the model from `ai/controller.py`
The model is a zip in `ai/models/` (the folder next to `ai/controller.py`). In your controller:

```python
from stable_baselines3 import PPO
import numpy as np, os
MODEL = PPO.load(os.path.join(os.path.dirname(os.path.abspath(__file__)), "models", "my_agent.zip"))

def decide_rl(values):
    obs_keys = [...]   # the SAME variables, in the SAME order as observation_names above
    obs = np.array([values[k] for k in obs_keys], dtype=np.float32)
    action, _ = MODEL.predict(obs, deterministic=True)
    return {"<your actuator name>": float(action[0])}
```

The observation vector must have the same length, order and **units** as in training (the FMU gives Joules for energy and degC for temperature; if your gateway rescales the values, convert them back).
The action is the value of the FMU command (here the heating setpoint, between `action_min` and `action_max`); the gateway clamps it again to your `min`/`max`.

## An alternative: learn from your own data
Reinforcement learning on the FMU is not the only way. The InfluxDB of your building already contains what the baseline (and your own controller) did:
you can train a policy by **imitation / offline learning** from that history (supervised learning of "action given observation", or offline RL), with no simulation in the loop.
No code is given here: it is a design choice you can make, and compare with the PPO agent.